In [10]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
import logging
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Union
warnings.filterwarnings('ignore')

In [11]:
class ProjectPaths:
    """專案路徑設定"""
    ROOT = Path("D:/Min/Python/Project/FA_Data")
    META_DATA = ROOT / "meta_data"
    DAILY_PRICE = ROOT / "daily_price"
    TECHNICAL_ANALYSIS = ROOT / "technical_analysis"
    
    # 資料檔案
    MARKET_INDEX = META_DATA / "market_index.csv"
    INDUSTRY_INDEX = META_DATA / "industry_index.csv"
    STOCK_DATA_WHOLE = META_DATA / "stock_data_whole.csv"
    ALL_STOCKS_DATA = META_DATA / "all_stocks_data.csv"

class DataConfig:
    """數據相關設定"""
    # 訓練測試分割
    TRAIN_TEST_SPLIT = 0.2
    RANDOM_SEED = 42
    
    # 日期格式
    DATE_FORMAT = "%Y-%m-%d"
    
    # 基本數據欄位
    BASIC_COLUMNS = [
        '證券代號', '證券名稱', '成交股數', '成交筆數', '成交金額',
        '開盤價', '最高價', '最低價', '收盤價', '漲跌(+/-)',
        '漲跌價差', '最後揭示買價', '最後揭示買量', '最後揭示賣價',
        '最後揭示賣量', '本益比', '日期'
    ]
    
    # 技術指標
    TECHNICAL_INDICATORS = {
        'TREND': ['SMA30', 'DEMA30', 'EMA30', 'TSF', 'middleband'],
        'MOMENTUM': ['RSI', 'MACD', 'MACD_signal', 'MACD_hist'],
        'VOLATILITY': ['slowk', 'slowd', 'SAR']
    }
    
    # 技術指標參數
    INDICATOR_PARAMS = {
        'SMA': {'timeperiod': 30},
        'DEMA': {'timeperiod': 30},
        'EMA': {'timeperiod': 30},
        'RSI': {'timeperiod': 14},
        'MACD': {'fastperiod': 12, 'slowperiod': 26, 'signalperiod': 9},
        'KD': {'fastk_period': 5, 'slowk_period': 3, 'slowd_period': 3},
        'TSF': {'timeperiod': 14},
        'BOLLINGER': {'timeperiod': 30},
        'SAR': {'acceleration': 0.02, 'maximum': 0.2}
    }

class ModelConfig:
    """模型相關設定"""
    RANDOM_SEED = 42
    N_ESTIMATORS = 100
    MAX_DEPTH = 5
    
    # 模型參數
    MODEL_PARAMS = {
        'random_forest': {
            'n_estimators': N_ESTIMATORS,
            'max_depth': MAX_DEPTH,
            'random_state': RANDOM_SEED
        }
    }

def setup_logging(log_level: int = logging.INFO) -> None:
    """設置日誌配置"""
    logging.basicConfig(
        level=log_level,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(
                ProjectPaths.ROOT / 'app.log',
                encoding='utf-8'
            )
        ]
    )

def format_number(num: Union[int, float]) -> str:
    """格式化數字輸出"""
    if isinstance(num, int):
        return f"{num:,}"
    return f"{num:,.2f}"

def display_stock_summary(df: pd.DataFrame) -> None:
    """顯示股票數據摘要"""
    print("\n=== 股票數據摘要 ===")
    print(f"股票代碼: {df['證券代號'].iloc[0]}")
    print(f"股票名稱: {df['證券名稱'].iloc[0]}")
    print(f"數據期間: {df['日期'].min().strftime('%Y-%m-%d')} 到 {df['日期'].max().strftime('%Y-%m-%d')}")
    print(f"交易天數: {len(df)} 天")
    
    print("\n--- 價格統計 ---")
    price_stats = {
        "平均收盤價": df['收盤價'].mean(),
        "最高價": df['最高價'].max(),
        "最低價": df['最低價'].min(),
        "價格波動範圍": df['最高價'].max() - df['最低價'].min(),
        "標準差": df['收盤價'].std()
    }
    
    for key, value in price_stats.items():
        print(f"{key}: {format_number(value)}")
    
    print("\n--- 交易統計 ---")
    trade_stats = {
        "平均日成交量": df['成交股數'].mean(),
        "平均日成交筆數": df['成交筆數'].mean(),
        "平均日成交金額": df['成交金額'].mean(),
    }
    
    for key, value in trade_stats.items():
        print(f"{key}: {format_number(value)}")

def load_stock_data(
    stock_id: str,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    verbose: bool = True
) -> pd.DataFrame:
    """載入股票數據"""
    try:
        data_path = ProjectPaths.ALL_STOCKS_DATA
        
        if not data_path.exists():
            raise FileNotFoundError(f"找不到數據文件: {data_path}")
            
        logging.info(f"正在讀取數據文件: {data_path}")
        df = pd.read_csv(data_path, low_memory=False)
        
        logging.info(f"原始數據形狀: {df.shape}")
        
        if df.empty:
            raise ValueError("讀取的數據框為空")
            
        df['證券代號'] = df['證券代號'].astype(str)
        df = df[df['證券代號'] == str(stock_id)]
        logging.info(f"過濾後的數據形狀 (股票 {stock_id}): {df.shape}")
        
        if df.empty:
            raise ValueError(f"找不到股票代碼 {stock_id} 的數據")
        
        df['日期'] = pd.to_datetime(df['日期'])
        
        if start_date:
            start_date = pd.to_datetime(start_date)
            df = df[df['日期'] >= start_date]
            logging.info(f"開始日期過濾後的數據形狀: {df.shape}")
            
        if end_date:
            end_date = pd.to_datetime(end_date)
            df = df[df['日期'] <= end_date]
            logging.info(f"結束日期過濾後的數據形狀: {df.shape}")
            
        df = df.sort_values('日期')
        
        if df.empty:
            raise ValueError("過濾後的數據框為空")
            
        if verbose:
            display_stock_summary(df)
            
            print("\n--- 技術指標概覽 ---")
            for category, indicators in DataConfig.TECHNICAL_INDICATORS.items():
                print(f"\n{category}類指標:")
                for indicator in indicators:
                    if indicator in df.columns:
                        print(f"{indicator:12} - 均值: {format_number(df[indicator].mean())}, "
                              f"標準差: {format_number(df[indicator].std())}")
        
        return df
        
    except Exception as e:
        logging.error(f"載入數據時出錯: {str(e)}")
        raise

def get_available_features() -> Dict[str, List[str]]:
    """獲取可用的特徵列表"""
    return {
        'basic': DataConfig.BASIC_COLUMNS,
        'technical': {
            category: indicators 
            for category, indicators in DataConfig.TECHNICAL_INDICATORS.items()
        }
    }

In [12]:
# 測試配置
if __name__ == "__main__":
    try:
        setup_logging()
        
        test_stock_id = '2330'
        test_start_date = '2023-01-01'
        test_end_date = '2024-01-01'
        
        print(f"\n正在載入台積電 ({test_stock_id}) 的數據...")
        
        df = load_stock_data(test_stock_id, test_start_date, test_end_date, verbose=True)
        
        print("\n=== 數據載入完成 ===")
        
    except Exception as e:
        print(f"錯誤: {str(e)}")

2024-10-29 17:02:26,680 - INFO - 正在讀取數據文件: D:\Min\Python\Project\FA_Data\meta_data\all_stocks_data.csv



正在載入台積電 (2330) 的數據...


2024-10-29 17:02:38,146 - INFO - 原始數據形狀: (2450900, 29)
2024-10-29 17:02:38,352 - INFO - 過濾後的數據形狀 (股票 2330): (2584, 29)
2024-10-29 17:02:38,357 - INFO - 開始日期過濾後的數據形狀: (437, 29)
2024-10-29 17:02:38,360 - INFO - 結束日期過濾後的數據形狀: (239, 29)



=== 股票數據摘要 ===
股票代碼: 2330
股票名稱: 台積電
數據期間: 2023-01-03 到 2023-12-29
交易天數: 239 天

--- 價格統計 ---
平均收盤價: 543.45
最高價: 594.00
最低價: 443.00
價格波動範圍: 151.00
標準差: 29.62

--- 交易統計 ---
平均日成交量: 27,535,566.40
平均日成交筆數: 29,635.65
平均日成交金額: 14,980,440,520.00

--- 技術指標概覽 ---

TREND類指標:
SMA30        - 均值: 536.47, 標準差: 28.85
DEMA30       - 均值: 543.00, 標準差: 29.66
EMA30        - 均值: 536.41, 標準差: 28.16
TSF          - 均值: 543.78, 標準差: 31.83
middleband   - 均值: 566.27, 標準差: 26.47

MOMENTUM類指標:
RSI          - 均值: 53.59, 標準差: 10.37
MACD         - 均值: 3.21, 標準差: 7.07
MACD_signal  - 均值: 3.12, 標準差: 6.46
MACD_hist    - 均值: 0.09, 標準差: 2.59

VOLATILITY類指標:
slowk        - 均值: 52.71, 標準差: 28.52
slowd        - 均值: 52.39, 標準差: 25.99
SAR          - 均值: 539.43, 標準差: 34.31

=== 數據載入完成 ===
